In [1]:
from time import time
import math, os
os.environ['OPENBLAS_NUM_THREADS'] = '2'
from sklearn import metrics
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.nn import Parameter
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import silhouette_score
from scMDC import scMultiCluster
import numpy as np
import collections
import h5py
import scanpy as sc
from preprocess import read_dataset, normalize, clr_normalize_each_cell
from utils import *
import timeit
import resource
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, adjusted_mutual_info_score
from sklearn.metrics import fowlkes_mallows_score, homogeneity_score, completeness_score, v_measure_score, silhouette_score

In [2]:
def eva(y_true, y_pred):
    nmi = nmi_score(y_true, y_pred)
    ari = ari_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    fmi = fowlkes_mallows_score(y_true, y_pred)
    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v = v_measure_score(y_true, y_pred)
    return nmi, ari, ami, fmi, hom, com, v

In [ ]:
import sys, timeit, argparse

def build_parser():
    parser = argparse.ArgumentParser(description='train',
                                     formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument('--n_clusters', default=27, type=int)
    parser.add_argument('--cutoff', default=0.5, type=float)
    parser.add_argument('--batch_size', default=256, type=int)
    parser.add_argument('--data_file', default=None)
    parser.add_argument('--maxiter', default=5000, type=int)
    parser.add_argument('--pretrain_epochs', default=400, type=int)
    parser.add_argument('--gamma', default=.1, type=float)
    parser.add_argument('--tau', default=1., type=float)
    parser.add_argument('--phi1', default=0.001, type=float)
    parser.add_argument('--phi2', default=0.001, type=float)
    parser.add_argument('--update_interval', default=1, type=int)
    parser.add_argument('--tol', default=0.001, type=float)
    parser.add_argument('--lr', default=1., type=float)
    parser.add_argument('--ae_weights', default=None)
    parser.add_argument('--save_dir', default='results/')
    parser.add_argument('--ae_weight_file', default='AE_weights_1.pth.tar')
    parser.add_argument('--resolution', default=0.2, type=float)
    parser.add_argument('--n_neighbors', default=30, type=int)
    parser.add_argument('--embedding_file', action='store_true', default=False)
    parser.add_argument('--prediction_file', action='store_true', default=False)
    parser.add_argument('-el','--encodeLayer', nargs='+', default=[256,64,32,16])
    parser.add_argument('-dl1','--decodeLayer1', nargs='+', default=[16,64,256])
    parser.add_argument('-dl2','--decodeLayer2', nargs='+', default=[16,20])
    parser.add_argument('--sigma1', default=2.5, type=float)
    parser.add_argument('--sigma2', default=1.5, type=float)
    parser.add_argument('--f1', default=1000, type=float)
    parser.add_argument('--f2', default=2000, type=float)
    parser.add_argument('--filter1', action='store_true', default=False)
    parser.add_argument('--filter2', action='store_true', default=False)
    parser.add_argument('--run', default=1, type=int)
    parser.add_argument('--device', default='cuda')
    return parser

if __name__ == "__main__":
    start = timeit.default_timer()
    parser = build_parser()
    # In notebooks, ignore Jupyter's argv
    if 'ipykernel' in sys.modules:
        args = parser.parse_args(args=[])   # <- key line
    else:
        args = parser.parse_args()
    print(args)


Namespace(n_clusters=27, cutoff=0.5, batch_size=256, data_file='/home/chengyue/data/multi-omics/spleen_lymph.h5', maxiter=5000, pretrain_epochs=400, gamma=0.1, tau=1.0, phi1=0.001, phi2=0.001, update_interval=1, tol=0.001, lr=1.0, ae_weights=None, save_dir='results/', ae_weight_file='AE_weights_1.pth.tar', resolution=0.2, n_neighbors=30, embedding_file=False, prediction_file=False, encodeLayer=[256, 64, 32, 16], decodeLayer1=[16, 64, 256], decodeLayer2=[16, 20], sigma1=2.5, sigma2=1.5, f1=1000, f2=2000, filter1=False, filter2=False, run=1, device='cuda')


In [ ]:
atac = sc.read_h5ad('./human_pbmc_3k_atac.h5ad')
rna = sc.read_h5ad('./human_pbmc_3k_rna.h5ad')
x2 = np.array(atac.X.toarray()).astype(np.float64)#.toarray()
x1= np.array(rna.X.toarray()).astype(np.float64)#.toarray()

import pandas as pd
cell_name = pd.read_csv('./human_pbmc_3k/label.csv', usecols=[1])
cell_type, y = np.unique(cell_name, return_inverse=True)

args.n_clusters=int(max(y)-min(y)+1)
print(args.n_clusters)
if args.filter1:
    importantGenes = geneSelection(x1, n=args.f1, plot=False)
    x1 = x1[:, importantGenes]
if args.filter2:
    importantGenes = geneSelection(x2, n=args.f2, plot=False)
    x2 = x2[:, importantGenes]

8


In [13]:
adata1 = sc.AnnData(x1)
adata1 = normalize(adata1,
                    size_factors=True,
                    normalize_input=True,
                    logtrans_input=True)

adata2 = sc.AnnData(x2)


AnnData object with n_obs × n_vars = 2711 × 25805
    obs: 'n_counts'
    var: 'n_counts'


In [14]:
input_size1 = adata1.n_vars
input_size2 = adata2.n_vars

In [15]:
encodeLayer = list(map(int, args.encodeLayer))
decodeLayer1 = list(map(int, args.decodeLayer1))
decodeLayer2 = list(map(int, args.decodeLayer2))

model = scMultiCluster(input_dim1=input_size1, input_dim2=input_size2, tau=args.tau,
                    encodeLayer=encodeLayer, decodeLayer1=decodeLayer1, decodeLayer2=decodeLayer2,
                    activation='elu', sigma1=args.sigma1, sigma2=args.sigma2, gamma=args.gamma, 
                    cutoff = args.cutoff, phi1=args.phi1, phi2=args.phi2, device=args.device).to(args.device)

In [16]:
model.pretrain_autoencoder(X1=adata1.X, X_raw1=adata1.raw.X, sf1=adata1.obs.size_factors,
        X2=adata2.X, X_raw2=adata2.X, sf2=adata1.obs.size_factors, batch_size=args.batch_size, 
        epochs=args.pretrain_epochs, ae_weights=args.ae_weight_file)###有改

/home/chengyue/scMDC-master/src/scMDC.py:157: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dataset = TensorDataset(torch.Tensor(X1), torch.Tensor(X_raw1), torch.Tensor(sf1), torch.Tensor(X2), torch.Tensor(X_raw2), torch.Tensor(sf2))


Pretraining stage
Pretrain epoch 1, Total loss:0.956817, ZINB loss1:0.465560, ZINB loss2:0.491258, KL loss:0.000000
Pretrain epoch 11, Total loss:0.793421, ZINB loss1:0.322062, ZINB loss2:0.471359, KL loss:0.000000
Pretrain epoch 21, Total loss:0.722554, ZINB loss1:0.268087, ZINB loss2:0.454466, KL loss:0.000000
Pretrain epoch 31, Total loss:0.690349, ZINB loss1:0.252172, ZINB loss2:0.438177, KL loss:0.000000
Pretrain epoch 41, Total loss:0.667217, ZINB loss1:0.245670, ZINB loss2:0.421548, KL loss:0.000000
Pretrain epoch 51, Total loss:0.647155, ZINB loss1:0.241797, ZINB loss2:0.405358, KL loss:0.000000
Pretrain epoch 61, Total loss:0.629510, ZINB loss1:0.239119, ZINB loss2:0.390391, KL loss:0.000000
Pretrain epoch 71, Total loss:0.613915, ZINB loss1:0.237108, ZINB loss2:0.376807, KL loss:0.000000
Pretrain epoch 81, Total loss:0.600365, ZINB loss1:0.235593, ZINB loss2:0.364772, KL loss:0.000000
Pretrain epoch 91, Total loss:0.588664, ZINB loss1:0.234335, ZINB loss2:0.354329, KL loss:0.

In [ ]:
#get k
latent = model.encodeBatch(torch.tensor(adata1.X).float().to(args.device), torch.tensor(adata2.X).float().to(args.device))
latent = latent.cpu().numpy()
if args.n_clusters == -1:
    n_clusters = GetCluster(latent, res=args.resolution, n=args.n_neighbors)
else:
    print("n_cluster is defined as " + str(args.n_clusters))
    n_clusters = args.n_clusters

y_pred, _, _, _, _ = model.fit(X1=adata1.X, X_raw1=adata1.raw.X, sf1=adata1.obs["size_factors"].values, 
    X2=adata2.X, X_raw2=adata2.raw.X, sf2=adata2.obs["size_factors"].values, y=y,
    n_clusters=n_clusters, batch_size=args.batch_size, num_epochs=args.maxiter, 
    update_interval=args.update_interval, tol=args.tol, lr=args.lr, save_dir=args.save_dir)


n_cluster is defined as 8
Clustering stage
Initializing cluster centers with kmeans.
Initializing k-means: AMI= 0.4639, NMI= 0.4664, ARI= 0.3200
Clustering   1: AMI= 0.4639, NMI= 0.4664, ARI= 0.3200
#Epoch 1: Total: 0.741943 Clustering Loss: 1.833974 ZINB Loss1: 0.246641 ZINB Loss2: 0.311742 KL Loss: 0.162653
Clustering   2: AMI= 0.4987, NMI= 0.4999, ARI= 0.3122
Clustering   3: AMI= 0.5253, NMI= 0.5268, ARI= 0.3148
Clustering   4: AMI= 0.5207, NMI= 0.5221, ARI= 0.2936
Clustering   5: AMI= 0.5207, NMI= 0.5222, ARI= 0.2803
Clustering   6: AMI= 0.5199, NMI= 0.5213, ARI= 0.2969
Clustering   7: AMI= 0.5177, NMI= 0.5192, ARI= 0.2758
Clustering   8: AMI= 0.5169, NMI= 0.5183, ARI= 0.3016
Clustering   9: AMI= 0.5160, NMI= 0.5174, ARI= 0.2722
Clustering   10: AMI= 0.5147, NMI= 0.5162, ARI= 0.2651
Clustering   11: AMI= 0.5192, NMI= 0.5206, ARI= 0.2880
#Epoch 11: Total: 0.558483 Clustering Loss: 0.248956 ZINB Loss1: 0.231556 ZINB Loss2: 0.301897 KL Loss: 0.134439
Clustering   12: AMI= 0.5160, NMI=

In [ ]:
final_latent = model.encodeBatch(torch.tensor(adata1.X).to(args.device), torch.tensor(adata2.X).to(args.device))
final_latent = final_latent.cpu().numpy()
y_pred_ = best_map(y, y_pred)
nmi_z, ari_z, ami_z, fmi_z, hom_z, com_z, v_z = eva(y, y_pred_)
print('z for clustering, NMI:{:.4f}, ARI:{:.4f}, AMI:{:.4f}, FMI:{:.4f}, HOM:{:.4f}, COM:{:.4f}, V:{:.4f}'.format(nmi_z, ari_z, ami_z, fmi_z, hom_z, com_z, v_z))

z for clustering, NMI:0.5179, ARI:0.2762, AMI:0.5165, FMI:0.4439, HOM:0.4351, COM:0.6396, V:0.5179
